# Anticipación en semanas 2, 4 y 6: utilidad operativa potencial

Preparado hasta que finalice la etapa `anticipacion`. Cada corte usa su propia cohorte elegible y conserva los folds por estudiante del día 14, sin condicionar pertenencia a estar activo al día 56. Se purgan estudiantes presentes en cualquier corte de 2014J sin usar sus outcomes.

Se comparan ambas familias finalistas con los mismos parámetros longitudinales fijados: solo estáticas, estáticas más indicador de registros y longitudinal completo. Es ablación de información en desarrollo, no tuning independiente de cada semana ni evidencia causal.


In [ ]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p/'config/protocolo-v2.json').is_file()), None)
if ROOT is None and (here/'pset-03/v2/config/protocolo-v2.json').is_file():
    ROOT = here/'pset-03/v2'
assert ROOT is not None, 'Abrir el notebook desde v2/notebooks o desde la raíz del repositorio'
os.environ['PSET3_V2_ROOT'] = str(ROOT)
sys.path.insert(0, str(ROOT))
from src.seguimiento import protocolo
from src.common import sha256
plan = protocolo()
def report(relative):
    path = ROOT/relative
    if not path.is_file():
        raise RuntimeError('Etapa pendiente: falta '+relative+'. Revisar docs/estado-v2.json y esperar al coordinador; no sustituir por resultados de v1.')
    return json.loads(path.read_text())
print('Protocolo verificado:', plan['id'])
print('Este notebook consulta resultados reales y reproduce sus cálculos; no inicia nuevas búsquedas ni llamadas de inferencia API.')


In [ ]:
early=report('docs/anticipacion/resultados.json')
assert early['status']=='complete' and early['protocol_id']==plan['id']
assert early['holdout_used'] is False and early['winner_changed'] is False
display(pd.DataFrame(early['cohorts']))
display(pd.DataFrame([{k:v for k,v in r.items() if k!='folds'} for r in early['models']]))


## Un presupuesto compartido entre rondas

K nace de la cohorte del día 14, por edición y fold. Se contrastan una sola ronda en semana 2, 4 o 6 y una política escalonada con 40/30/30 % del mismo presupuesto. No se repiten matrículas; también se informan personas únicas. Se muestran escenarios de 7 días de coordinación y 14 de apoyo, con plazos ilustrativos en los días 56 y 70. No son políticas institucionales validadas.


In [ ]:
policies=pd.DataFrame([{k:v for k,v in p.items() if k!='rounds'} for p in early['policies']])
assert (policies['used']<=policies['budget']).all()
display(policies)
print('Los TP con margen son candidatos potencialmente atendibles bajo los supuestos, no fracasos evitados por una tutoría.')


## Ausencia de actividad: asociación e incremento predictivo

Una tasa alta de Fail en el grupo sin registros no basta para demostrar utilidad incremental. Se muestran diferencia de riesgo, razón de riesgos, correlación binaria phi, intervalos por remuestreo de estudiante y estandarización por edición. El incremento predictivo se evalúa separadamente en la comparación estáticas / estáticas+indicador / longitudinal. Estandarizar no demuestra ausencia de confusores ni causalidad.


In [ ]:
display(pd.DataFrame(early['associations']))
from IPython.display import display, Markdown
for limitation in early['limits']:
    display(Markdown('- '+limitation))
